In [1]:
import polars as pl
from procompa import get_project_root

PRJ_ROOT = get_project_root()
data_dir = PRJ_ROOT / "data"

## Input for homomultimer comparison pooled vs pair modles

get all proteins and their sequence

In [15]:
protein_AF_info = pl.read_parquet(data_dir/ "Homomultimer/Pipeline_prep/proteins.parquet") #mapping of pdb id to protein

In [ ]:
with open(data_dir/ "iPTM_and_pLDDT/all_yeast_proteins_uniprot_mapped_sequences.csv", "w") as f:
    for uid, seq in protein_AF_info.select(["uniprot_id", "seq"]).iter_rows():
        f.write(f"{uid},{seq}\n")

After finding homomultimers

In [10]:
homomultimers = pl.read_csv(data_dir/ "Homomultimer/Pipeline_prep/homomultimers.csv") #mapping of pdb id to protein

In [11]:
#filter out matches,that potentially have Fusion tags, Affinity tags and cloning linker residues, or Partner proteins
homomultimers = homomultimers.filter((~pl.col("uniprot_id").str.contains(";") )& (pl.col("uniprot_seq_len") >= pl.col("seq_len")))

In [12]:
# keep match with best coverage
homomultimers = homomultimers.with_columns(
    coverage = (pl.col("uniprot_seq_len") / pl.col("seq_len"))
)

homomultimers = homomultimers.sort("coverage", descending=True).unique(subset=["uniprot_id"], keep="first")

In [13]:
homomultimers_CF = homomultimers.select([
    pl.format("HOMO_{}", pl.col("uniprot_id")).alias("#Complex ac"),
    pl.format(
        "{}({})", pl.col("uniprot_id"), pl.col("n_chains").cast(pl.Int64)
    ).alias("Identifiers (and stoichiometry) of molecules in complex"),
    pl.format("{} homomultimer", pl.col("uniprot_id")).alias(
        "Recommended name"
    ),
    pl.col("pdb_id"),
    pl.col("n_chains").cast(pl.Int64),
    pl.col("coverage").round(3),
])

In [ ]:
# 4. Write input for CF pipeline to TSV
homomultimers_CF.write_csv(data_dir/"Pipeline/6_sixth_subset_homomultimers_pool_vs_pair/sixth_input_homomultimers_pool_vs_pair.tsv", separator="\t")

Wrote 494 homomultimer complexes


Get sequences for all proteins that i have Homomultimer pdb files for 

In [ ]:
'''
find prot which are not :/cluster/project/beltrao/kdammer/master_thesis/data/iPTM_and_pLDDT/all_yeast_proteins_uniprot_mapped_sequences.csv
add them to csv (so get sequences)
create dataframe in correct input format:Complex ac = HOMO_<uniprot_id> (synthetic ID)
Identifiers (and stoichiometry) of molecules in complex = <uniprot_id>(0) (let Stoic predict freely) or <uniprot_id>(n_chains) 

'''


First run of benchmark (exact pdb macth found)

In [2]:
#complexes for which a pair is missing

CP_AF_model_exist_mapping = pl.read_parquet(data_dir/ "iPTM_and_pLDDT/final_CP_YM_complexes_pairs_AF_model_exists.parquet")

CP_AF_model_exist_mapping= (
    CP_AF_model_exist_mapping
    .filter(pl.col("AF_model_exists") == False)
    .select(pl.col("complex_ac").str.split("|"))
    .explode("complex_ac")
    .unique()
    .with_columns(pl.col("complex_ac").str.strip_chars())   
)
#complexes which have an exact pdb match
Cpx_match_class = pl.read_csv(data_dir/ 'complete_complex_pdb_mapping_v2/all_pdb_matches_with_match_class.csv')

#Cpx_match_class
Cpx_exact_match = (
    Cpx_match_class
    .filter(pl.col("match_class")== "exact_pdb_match")
    .with_columns(pl.col("complex_ac").str.strip_chars())  
)

#get complexes with exact match classes, that have all pairs
Cpx_exact_match_with_all_pairs = Cpx_exact_match.filter(
    ~pl.col('complex_ac').is_in(CP_AF_model_exist_mapping['complex_ac'].to_list())  
)

CP_data = pl.read_csv(data_dir/"Complex_Portal/Saccharomyces_cerevisiae_ComplexTab.tsv", separator="\t")

sumbit_benchmark_first_half = CP_data.filter(
    pl.col("#Complex ac").is_in(Cpx_exact_match_with_all_pairs['complex_ac'].to_list())   
)

In [3]:
sumbit_benchmark_first_half.write_csv(data_dir/"Pipeline/7_benchmark_part_one/seventh_input_benchmark_first_half.tsv", separator="\t")

preparation for second part benchamrk (once all heterodimer pairs ar eneed are run)

In [4]:
Cpx_exact_match_without_pairs = Cpx_exact_match.filter(
    pl.col('complex_ac').is_in(CP_AF_model_exist_mapping['complex_ac'].to_list())
)

sumbit_benchmark_second_round = CP_data.filter(
    pl.col("#Complex ac").is_in(Cpx_exact_match_without_pairs['complex_ac'].to_list())
)

In [5]:
sumbit_benchmark_second_round.write_csv(data_dir/"Pipeline/8_benchmark_part_two/eighth_input_benchmark_second_half.tsv", separator="\t")

In [6]:
benchmark_concat = pl.concat([sumbit_benchmark_first_half, sumbit_benchmark_second_round])

In [7]:
set1 = set(benchmark_concat['#Complex ac'])
set2 = set(Cpx_exact_match['complex_ac'])

if set1 == set2:
    print("All complex IDs match")
else:
    print("Mismatch")

All complex IDs match


## Input 9 Benchmark no mmseq homology match

In [ ]:
# find complexes which have no homology match in PDB
CP_pdb_mapping = pl.read_csv(data_dir/"complete_complex_pdb_mapping_v2/all_pdb_matches_with_match_class.csv")
no_match_CP_pdb_mapping = CP_pdb_mapping.filter(pl.col("match_class") == "no_homology_match")
CP_df = pl.read_csv(data_dir/"Complex_Portal/Saccharomyces_cerevisiae_ComplexTab.tsv", separator="\t")

In [ ]:
input_9_benchmark = CP_df.filter(pl.col("#Complex ac").is_in(no_match_CP_pdb_mapping['complex_ac'].to_list()))

In [8]:
input_9_benchmark.write_csv("/cluster/project/beltrao/kdammer/master_thesis/data/Pipeline/9_complexes_no_mmseq_homology_match/ninth_input_benchmark_no_homology_match.tsv", separator="\t")